# Multilayer Coupling Schemes: spread vs. couple-to-self

Building the state network for a multilayer network means choosing **what an inter-layer relaxation link points to**. Infomap gives you two schemes:

- **Spread to neighbours (default).** On relaxation from `(layer1, n)`, the walker moves to a *neighbour* of `n` in the target layer. One inter-layer link per out-neighbour per layer pair.
- **Couple physical nodes only (`--multilayer-relax-to-self`).** On relaxation, the walker moves to the *same physical node* `(layer2, n)`. One inter-layer link per reachable layer.

to-self builds a far smaller state network. But which scheme finds *better* communities is not one-sided: **sometimes to-self is better, sometimes spread is**, and there is a clear rule for which. This notebook works out that rule so you can choose deliberately.

## Why the two differ

The relax step lands on a neighbour and is independent of the source layer:

$$ p_{(i,n)\to(j,m)} = (1-r)\,\delta_{ij}\frac{w^i_{nm}}{s_i(n)} + r\,\frac{w^j_{nm}}{S_n} $$

The two schemes differ only in the inter-layer (relax) step, and they trade off opposite strengths:

- **Spread** relaxes to a *neighbour* in the other layer. That neighbour shares the node's community with probability `1 - mu` (where `mu` is the fraction of cross-community links), so the inter-step **pools community evidence across layers** (it rescues weak per-layer structure) but also **injects cross-community noise at rate `mu`** (it hurts at heavy mixing).
- **to-self** relaxes to the *same node*, which is always in-community, so it adds **no cross-community noise** to the inter-step but it **cannot pool** neighbour evidence either.

So to-self trades pooling power for noise immunity. That single trade-off explains everything below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_mutual_info_score

from infomap import Infomap

%matplotlib inline


def make_shared(n_per_comm, K, L, within_deg, cross_deg, seed=0):
    """L layers sharing the same K planted communities (independent edge draws)."""
    rng = np.random.default_rng(seed)
    N = n_per_comm * K
    p_in = min(1.0, within_deg / max(1, n_per_comm - 1))
    p_out = cross_deg / max(1, (K - 1) * n_per_comm)
    comm = {nd: (nd - 1) // n_per_comm for nd in range(1, N + 1)}
    gt, intra = {}, []
    for layer in range(1, L + 1):
        for nd in range(1, N + 1):
            gt[(layer, nd)] = comm[nd]
        for i in range(1, N + 1):
            for j in range(i + 1, N + 1):
                if rng.random() < (p_in if comm[i] == comm[j] else p_out):
                    intra.append((layer, i, j, 1.0))
    return intra, gt


def run_scheme(intra, *, to_self, seed=123, num_trials=5):
    im = Infomap(silent=True, seed=seed, num_trials=num_trials,
                 multilayer_relax_to_self=to_self)
    for link in intra:
        im.add_multilayer_intra_link(*link)
    im.run()
    part = {(n.layer_id, n.node_id): n.module_id for n in im.nodes}
    return part, im.num_links


def ami(part, gt):
    keys = sorted(set(part) & set(gt))
    return adjusted_mutual_info_score([gt[k] for k in keys], [part[k] for k in keys])

## The cost: state-network size

Spread materializes one inter-layer link per out-neighbour per layer pair, `O(L^2 * k)`. to-self adds one diagonal link per reachable layer, `O(L * k)`. The gap widens with the number of layers `L`, which is what makes to-self attractive for large or many-layer (temporal) networks.

In [ ]:
layer_counts = [2, 4, 6, 8, 10]
links_spread, links_self = [], []
for L in layer_counts:
    intra, _ = make_shared(n_per_comm=40, K=3, L=L, within_deg=10, cross_deg=1, seed=1)
    links_spread.append(run_scheme(intra, to_self=False, num_trials=1)[1])
    links_self.append(run_scheme(intra, to_self=True, num_trials=1)[1])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(layer_counts, links_spread, 'o-', label='spread (default)')
ax.plot(layer_counts, links_self, 's-', label='to-self')
ax.set_xlabel('number of layers L')
ax.set_ylabel('state-network links')
ax.set_title('State-network size vs number of layers')
ax.legend()
fig.tight_layout()

## When does each scheme win?

We sweep the mixing `mu` (more cross-community links = harder) in two regimes and measure the AMI of each scheme's partition against the planted communities, averaged over seeds:

- **Dense, self-sufficient communities** (each layer detectable on its own).
- **Sparse communities** (too weak per layer, so detection relies on pooling across layers).

The winner flips between the two regimes.

In [ ]:
def sweep(n_per_comm, within_deg, cross_list, seeds=(1, 2)):
    rows = []
    for cd in cross_list:
        mu = cd / (within_deg + cd)
        agg = {False: [], True: []}
        for s in seeds:
            intra, gt = make_shared(n_per_comm, 4, 4, within_deg, cd, s)
            for to_self in (False, True):
                part, _ = run_scheme(intra, to_self=to_self)
                agg[to_self].append(ami(part, gt))
        for to_self in (False, True):
            rows.append({'mu': round(mu, 2), 'scheme': 'to-self' if to_self else 'spread',
                         'AMI': float(np.mean(agg[to_self]))})
    return pd.DataFrame(rows)

dense = sweep(n_per_comm=120, within_deg=16, cross_list=[8, 12, 16])
sparse = sweep(n_per_comm=400, within_deg=10, cross_list=[2, 4, 6])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, df, title in [(axes[0], dense, 'Dense, self-sufficient communities'),
                      (axes[1], sparse, 'Sparse communities (need pooling)')]:
    for scheme, g in df.groupby('scheme'):
        ax.plot(g['mu'], g['AMI'], 'o-', label=scheme)
    ax.set_xlabel('mixing mu')
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.05)
    ax.legend()
axes[0].set_ylabel('AMI vs planted communities')
fig.tight_layout()

Reading the two panels:

- **Dense (left):** as mixing rises, **spread collapses first** while to-self still recovers the communities. With a self-sufficient per-layer signal, spread's gain from pooling is small and its cross-community noise dominates, so **to-self wins** (and it is cheaper).
- **Sparse (right):** **to-self collapses first** while spread holds. Here the per-layer signal is too weak alone, so spread's pooling is essential and to-self under-couples. **Spread wins.**
- At low mixing both recover the communities (a tie), and to-self does it with far fewer links.

Relax limits (`--multilayer-relax-limit`) retune how the coupling concentrates across layers but do not move this boundary; the driver is per-layer detectability and mixing, not the number of layers reached.

## Decision rule

- **Use `--multilayer-relax-to-self`** when each layer's community structure is detectable on its own (dense communities, or many thin layers as in temporal networks). There to-self matches or **beats** spread, is **more robust to inter-community mixing**, and builds a 3-6x smaller state network with lower memory and runtime, with the saving growing in the number of layers.
- **Keep the default spread** when the per-layer structure is sparse and you rely on the multilayer model to pool weak evidence across layers. There to-self under-couples and can collapse, so the cheaper network would cost you the communities.
- The default stays spread. Enable diagonal coupling with the `--multilayer-relax-to-self` CLI flag or `Infomap(multilayer_relax_to_self=True)`.